# 🚀 SuperAI V11 — Google Colab Notebook

**Complete guide: Setup → Launch → Test all features**

### What's in V11:
- **V10 base**: 11 features (Reflection, Learning, Memory, Parallel Agents, RAG++, etc.)
- **Step 1**: True RLHF System (Reward Model + DPO + GRPO training)
- **Step 2**: Tool Calling Engine (web_search, calculator, code_execute, wikipedia, weather)
- **Step 3**: Multi-Model Consensus (parallel models + voting + conflict detection)

> **Runtime**: Runtime → Change runtime type → **T4 GPU** → Save

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Check GPU
# ═══════════════════════════════════════════════════════════
import subprocess
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(f'✅ GPU detected: {r.stdout.strip()}')
else:
    print('⚠️  No GPU — CPU mode (slower inference)')

import psutil
vm = psutil.virtual_memory()
print(f'📊 RAM: {vm.total // (1024**3)} GB total, {vm.available // (1024**3)} GB available')

import shutil
free = shutil.disk_usage('/').free // (1024**3)
print(f'💾 Disk: {free} GB free')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Upload & Extract superai_v11_complete.zip
# ═══════════════════════════════════════════════════════════
from google.colab import files
print('⬆️  Select superai_v11_complete.zip to upload...')
uploaded = files.upload()

import zipfile, os
zip_name = list(uploaded.keys())[0]
print(f'\n📦 Extracting {zip_name}...')
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/')

# Find the extracted folder
import glob
candidates = glob.glob('/content/superai_v11*')
PROJECT_DIR = candidates[0] if candidates else '/content/superai_v11_final'
print(f'✅ Project at: {PROJECT_DIR}')
print(f'   Python files: {len(list(__import__("pathlib").Path(PROJECT_DIR).rglob("*.py")))}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — ONE-COMMAND LAUNCH
# ═══════════════════════════════════════════════════════════
NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'  # ← paste your token from ngrok.com

import subprocess, sys
result = subprocess.run(
    [sys.executable, f'{PROJECT_DIR}/scripts/run_colab_v11.py',
     '--token', NGROK_TOKEN,
     '--model', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'],
    # To enable only specific V11 features to save VRAM:
    # '--features', 'F5,F8,S1,S2'
)
# The public URL is stored in os.environ['SUPERAI_V11_URL'] after launch

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Configure BASE_URL for all test cells
# ═══════════════════════════════════════════════════════════
import os, requests

BASE = os.environ.get('SUPERAI_V11_URL', 'http://localhost:8000')
print(f'🌐 Base URL: {BASE}')

def api(method, path, **kwargs):
    """Helper for API calls with nice output."""
    url = f'{BASE}{path}'
    r   = getattr(requests, method)(url, timeout=120, **kwargs)
    return r.json()

# Health check
h = api('get', '/health')
print(f'\n✅ Health: {h}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Basic Chat Test
# ═══════════════════════════════════════════════════════════
print('💬 Testing basic chat...')
r = api('post', '/api/v1/chat/',
    json={'prompt': 'Hello SuperAI V11! What new features do you have?', 'max_tokens': 150})

if r.get('success'):
    d = r['data']
    print(f'\n🤖 Answer: {d["answer"][:300]}')
    print(f'   Model: {d["model_used"]} | Task: {d["task_type"]} | {d["latency_ms"]:.0f}ms')
else:
    print(f'❌ Error: {r}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — System Status
# ═══════════════════════════════════════════════════════════
r = api('get', '/api/v1/system/status')
d = r.get('data', {})
print('📊 System Status:')
print(f'  Version:    {d.get("version")}')
print(f'  CPU:        {d.get("cpu_pct")}%')
print(f'  RAM:        {d.get("ram_pct")}%')
print(f'  GPU:        {d.get("gpu_info") or "N/A"}')
print(f'  Uptime:     {d.get("uptime_s")}s')
print(f'  Requests:   {d.get("requests_total")}')
print(f'  Avg latency:{d.get("avg_latency_ms")}ms')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — V11 Step 1: RLHF System
# ═══════════════════════════════════════════════════════════
print('=' * 55)
print('  V11 STEP 1: RLHF System')
print('=' * 55)

# Check RLHF status
r = api('get', '/api/v1/rlhf/status')
if r.get('success'):
    d = r['data']
    print(f'\n✅ RLHF Pipeline loaded')
    print(f'   Runs total:   {d.get("runs_total")}')
    print(f'   DPO running:  {d.get("dpo_busy")}')
    rm = d.get('reward_model', {})
    print(f'   Reward model: neural={rm.get("use_neural")}, trained={rm.get("trained")}')
else:
    print(f'⚠️  RLHF: {r.get("error", "not loaded")}')

print()

# Score a response with reward model
r = api('post', '/api/v1/rlhf/score',
    params={'prompt': 'What is machine learning?',
            'response': 'Machine learning is a subset of AI that enables computers to learn '
                        'from data. For example, neural networks can specifically learn '
                        'image recognition patterns step by step.'})
if r.get('success'):
    print(f'⭐ Reward score: {r["data"]["score"]:.3f}  (range: -1.0 to +1.0)')
else:
    print(f'⚠️  Score: {r.get("error")}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — RLHF: Train Reward Model (from feedback DB)
# ═══════════════════════════════════════════════════════════
print('🧠 Training neural reward model from feedback data...')
print('(Requires feedback data — will show "not enough pairs" if no ratings yet)')

r = api('post', '/api/v1/rlhf/reward-model/train')
print(f'\nResult: {r.get("data") or r.get("error")}')

print()
print('📝 To trigger DPO training (requires trl + peft + ratings):')
print('   POST /api/v1/rlhf/train/dpo  {"model_name": "TinyLlama/...", "epochs": 2}')
print('   This runs in background — check status at GET /api/v1/rlhf/status')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — V11 Step 2: Tool Calling Engine
# ═══════════════════════════════════════════════════════════
print('=' * 55)
print('  V11 STEP 2: Tool Calling Engine')
print('=' * 55)

# List available tools
r = api('get', '/api/v1/tools/list')
if r.get('success'):
    tools = r['data']['tools']
    print(f'\n🛠️  Available tools ({len(tools)}):' )
    for t in tools:
        safe_label = '✅' if t['safe'] else '⚠️'
        print(f'  {safe_label} {t["name"]:15s} [{t["category"]}] — {t["description"][:50]}')
else:
    print(f'⚠️  Tools: {r.get("error")}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — Tool Calling: Execute tools directly
# ═══════════════════════════════════════════════════════════

# Test calculator
print('🔢 Calculator tool:')
r = api('post', '/api/v1/tools/execute',
    json={'tool_name': 'calculator', 'arguments': {'expression': '(15 * 8) + sqrt(144)'}})
if r.get('success'):
    print(f'   {r["data"]["output"]}  ({r["data"]["exec_ms"]:.1f}ms)')

print()

# Test datetime
print('🕐 DateTime tool:')
r = api('post', '/api/v1/tools/execute',
    json={'tool_name': 'datetime', 'arguments': {}})
if r.get('success'):
    print(f'   {r["data"]["output"]}')

print()

# Test code execution
print('⚙️  Code execution tool:')
r = api('post', '/api/v1/tools/execute',
    json={'tool_name': 'code_execute',
          'arguments': {'code': 'numbers = [1,2,3,4,5]\nprint(f"Sum: {sum(numbers)}")\nprint(f"Mean: {sum(numbers)/len(numbers):.2f}")',
                        'language': 'python'}})
if r.get('success'):
    print(f'   {r["data"]["output"]}')
else:
    print(f'   {r.get("error")}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — Tool Calling: Full pipeline (prompt → tools → enriched answer)
# ═══════════════════════════════════════════════════════════
print('🔗 Full tool calling pipeline:')

test_prompts = [
    'Calculate compound interest: 10000 * (1 + 0.07) ** 10',
    'What is the current date and time?',
]

for prompt in test_prompts:
    print(f'\nPrompt: "{prompt}"')
    r = api('post', '/api/v1/tools/call',
        json={'prompt': prompt, 'autonomy_level': 2, 'max_tools': 2})
    if r.get('success'):
        d = r['data']
        print(f'  Tools used: {d["tools_used"]}')
        for tr in d.get('tool_results', []):
            print(f'  [{tr["tool"]}] {tr["output"][:100]}')
        print(f'  Total: {d["total_ms"]:.1f}ms')
    else:
        print(f'  Error: {r.get("error")}')

print()

# Tool-enriched chat (tools injected into prompt automatically)
print('💬 Tool-enriched chat (auto tools in pipeline):')
r = api('post', '/api/v1/chat/',
    json={'prompt': 'What is 2 to the power of 10?', 'max_tokens': 100})
if r.get('success'):
    print(f'  {r["data"]["answer"][:200]}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12 — Web Search tool (requires internet)
# ═══════════════════════════════════════════════════════════
print('🌐 Web Search tool (live internet):')
r = api('post', '/api/v1/tools/execute',
    json={'tool_name': 'web_search',
          'arguments': {'query': 'latest AI breakthroughs 2025', 'max_results': 3}})
if r.get('success'):
    output = r['data']['output']
    print(output[:600])
    print(f'\n⏱️  Search took {r["data"]["exec_ms"]:.0f}ms')
else:
    print(f'⚠️  {r.get("error")} (internet required)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 13 — V11 Step 3: Multi-Model Consensus
# ═══════════════════════════════════════════════════════════
print('=' * 55)
print('  V11 STEP 3: Multi-Model Consensus')
print('=' * 55)

r = api('get', '/api/v1/consensus/status')
if r.get('success'):
    d = r['data']
    print(f'\n✅ Consensus Engine loaded')
    print(f'   Models:   {d.get("models")}')
    print(f'   Strategy: {d.get("strategy")}')
    print(f'   Conflicts:{d.get("total_conflicts")}')
    print(f'   Meta-eval:{d.get("use_meta")}')
elif 'need >= 2 models' in str(r.get('error','')):
    print('\n⚠️  Consensus requires >= 2 models in config.yaml:')
    print('   Edit config/consensus/models: ["model_a", "model_b"]')
    print('   Then restart server')
    print()
    print('💡 For demo, testing consensus API directly...')
else:
    print(f'   Status: {r.get("error")}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 14 — V10 Features Still Working (regression check)
# ═══════════════════════════════════════════════════════════
print('=' * 55)
print('  V10 Feature Regression Check')
print('=' * 55)

checks = [
    ('F1 Self-Reflection',  'get',  '/api/v1/intelligence/reflect', None),
    ('F5 RAG++ Knowledge',  'post', '/api/v1/knowledge/retrieve',
     {'query': 'what is FastAPI'}),
    ('F6 Self-Improvement', 'get',  '/api/v1/intelligence/improve/stats', None),
    ('F7 Model Registry',   'get',  '/api/v1/intelligence/registry', None),
    ('F8 AI Security',      'post', '/api/v1/security/assess',
     {'prompt': 'Hello, how are you?', 'session_id': 'test'}),
    ('F11 Personality',     'get',  '/api/v1/personality/profile', None),
    ('F2 Learning',         'get',  '/api/v1/learning/status', None),
]

for name, method, path, body in checks:
    try:
        kwargs = {'json': body} if body else {}
        r = api(method, path, **kwargs)
        ok = r.get('success', False)
        status = '✅' if ok else '⚠️ '
        detail = '' if ok else f"  → {r.get('error','')}"
        print(f'  {status} {name}{detail}')
    except Exception as e:
        print(f'  ❌ {name}: {str(e)[:60]}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 15 — Unit Tests (run directly in Colab)
# ═══════════════════════════════════════════════════════════
import subprocess, sys

print('🧪 Running V11 unit tests...')
result = subprocess.run(
    [sys.executable, '-m', 'pytest',
     'tests/unit/test_v11_rlhf.py',
     'tests/unit/test_v11_tools.py',
     'tests/unit/test_v11_consensus.py',
     '-v', '--tb=short', '-q', '--asyncio-mode=auto'],
    cwd=PROJECT_DIR,
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-500:])

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 16 — Full V11 Integration Test
# ═══════════════════════════════════════════════════════════
print('🔄 Full V11 Integration Test')
print('Testing: Chat → Tools → RLHF score → Feedback → Improvement')
print()

# Step 1: Send a chat message
chat_r = api('post', '/api/v1/chat/',
    json={'prompt': 'What is the capital of Japan and its population?',
          'max_tokens': 100, 'session_id': 'integration-test'})
response_id = None
if chat_r.get('success'):
    d = chat_r['data']
    response_id = d.get('response_id')
    print(f'1️⃣  Chat answer: {d["answer"][:150]}')
    print(f'   Response ID: {response_id}')

print()

# Step 2: Score with RLHF reward model
if response_id:
    score_r = api('post', '/api/v1/rlhf/score',
        params={'prompt': 'What is the capital of Japan?',
                'response': chat_r['data']['answer']})
    if score_r.get('success'):
        print(f'2️⃣  RLHF Reward score: {score_r["data"]["score"]:.3f}')

print()

# Step 3: Give feedback (simulate user rating)
if response_id:
    fb_r = api('post', '/api/v1/feedback/',
        json={'response_id': response_id, 'score': 4,
              'comment': 'Good answer, clear and concise',
              'session_id': 'integration-test'})
    if fb_r.get('success'):
        print(f'3️⃣  Feedback recorded: ⭐⭐⭐⭐ (4/5)')

print()

# Step 4: Check improvement stats
imp_r = api('get', '/api/v1/intelligence/improve/stats')
if imp_r.get('success'):
    d = imp_r['data']
    print(f'4️⃣  Self-improvement stats: {d.get("failures_7d",0)} failures tracked')

print()
print('✅ Integration test complete!')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 17 — Streaming Chat (WebSocket)
# ═══════════════════════════════════════════════════════════
import json, asyncio

async def stream_chat(prompt: str, base_url: str):
    try:
        import websockets
        ws_url = base_url.replace('https://','wss://').replace('http://','ws://')
        ws_url = f'{ws_url}/ws/chat'
        print(f'WebSocket: {ws_url}')
        async with websockets.connect(ws_url, close_timeout=30) as ws:
            await ws.send(json.dumps({'type':'chat','payload':{'prompt':prompt,'max_tokens':100}}))
            print('Response: ', end='', flush=True)
            while True:
                msg = json.loads(await asyncio.wait_for(ws.recv(), timeout=60))
                if msg['type'] == 'token':
                    print(msg['data'], end='', flush=True)
                elif msg['type'] == 'done':
                    print(f'\n✅ Done ({msg["data"].get("latency_ms",0):.0f}ms)')
                    break
                elif msg['type'] == 'error':
                    print(f'\n❌ {msg["data"]}')
                    break
    except Exception as e:
        print(f'\n⚠️  WebSocket: {e}')
        print('   (Try: pip install websockets)')

print('📡 Testing WebSocket streaming...')
await stream_chat('Explain what makes Python great for AI in 2 sentences.', BASE)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 18 — View server logs
# ═══════════════════════════════════════════════════════════
log_path = '/content/superai_v11_data/logs/server.log'
try:
    with open(log_path) as f:
        lines = f.readlines()
    print(f'📋 Last 30 log lines ({len(lines)} total):')
    print(''.join(lines[-30:]))
except FileNotFoundError:
    print(f'⚠️  Log file not found: {log_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 19 — Complete curl cheatsheet
# ═══════════════════════════════════════════════════════════
import os
BASE = os.environ.get('SUPERAI_V11_URL', 'YOUR_URL')

cheatsheet = f"""# SuperAI V11 — curl Cheatsheet
# BASE = {BASE}

# ── Core ─────────────────────────────────────────────────────
curl {BASE}/health
curl {BASE}/docs   # Interactive Swagger UI

# ── Chat ─────────────────────────────────────────────────────
curl -X POST {BASE}/api/v1/chat/ \\
  -H 'Content-Type: application/json' \\
  -d '{{"prompt":"Hello V11!","max_tokens":128}}'

# ── V11 Step 1: RLHF ─────────────────────────────────────────
curl {BASE}/api/v1/rlhf/status
curl -X POST {BASE}/api/v1/rlhf/score \\
  -G --data-urlencode 'prompt=What is AI?' \\
     --data-urlencode 'response=AI is machine intelligence.'
curl -X POST {BASE}/api/v1/rlhf/train/dpo \\
  -H 'Content-Type: application/json' \\
  -d '{{"model_name":"TinyLlama/TinyLlama-1.1B-Chat-v1.0","epochs":2}}'

# ── V11 Step 2: Tools ─────────────────────────────────────────
curl {BASE}/api/v1/tools/list
curl -X POST {BASE}/api/v1/tools/execute \\
  -H 'Content-Type: application/json' \\
  -d '{{"tool_name":"calculator","arguments":{{"expression":"sqrt(144)"}}}}'  
curl -X POST {BASE}/api/v1/tools/call \\
  -H 'Content-Type: application/json' \\
  -d '{{"prompt":"what time is it now","max_tools":2}}'

# ── V11 Step 3: Consensus ─────────────────────────────────────
curl {BASE}/api/v1/consensus/status
curl -X POST {BASE}/api/v1/consensus/run \\
  -H 'Content-Type: application/json' \\
  -d '{{"prompt":"What is quantum computing?","strategy":"best"}}'

# ── Feedback ─────────────────────────────────────────────────
curl -X POST {BASE}/api/v1/feedback/ \\
  -H 'Content-Type: application/json' \\
  -d '{{"response_id":"YOUR_ID","score":5,"comment":"Excellent!"}}'
"""
print(cheatsheet)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 20 — Stop server
# ═══════════════════════════════════════════════════════════
import subprocess
try:
    from pyngrok import ngrok
    ngrok.kill()
    print('✅ ngrok tunnels closed')
except Exception:
    pass
subprocess.run(['pkill', '-f', 'uvicorn'], capture_output=True)
print('✅ Server stopped')